# 01 · Acquire

This notebook downloads every source and saves it in `data/raw/`. A file that is already in `data/raw/` is not downloaded again.

Every real download is added to `data/raw/download_log.csv` with the full URL, the HTTP status, the size of the file and the time in UTC.
The log is never rebuilt: each run adds its downloads to the rows that are already there, so it keeps the history of when each file was
actually fetched.

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)
LOG_FILE = RAW / "download_log.csv"

We create one session with retries and a timeout, and a `User-Agent` that says who we are. Every source then goes through the same
`download` function, which skips files that are already on disk and logs every request it really makes, including failed ones.

In [2]:
retry = Retry(total=5, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
session = requests.Session()
session.mount("http://", HTTPAdapter(max_retries=retry))
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers["User-Agent"] = (
    "ESMT MAAI data wrangling project, group 4 "
    "(https://github.com/GuillermoMorfin/esmt-data-wrangling-project)"
)

LOG_COLUMNS = ["source", "file", "url", "status", "bytes", "downloaded_at_utc", "note"]
new_downloads = []


def download(source, url, path, params=None):
    """Save `url` to `path` unless the file is already there, and log the request.

    Returns True if the file is on disk afterwards. Files that are already on disk are not
    requested or logged again, so the log keeps the time each file was really downloaded.
    """
    if path.exists():
        print(f"{path.name}: already in data/raw, not downloaded again")
        return True

    downloaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    try:
        r = session.get(url, params=params, timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        response = getattr(e, "response", None)
        new_downloads.append({
            "source": source,
            "file": path.name,
            "url": response.url if response is not None else url,
            "status": response.status_code if response is not None else type(e).__name__,
            "bytes": 0,
            "downloaded_at_utc": downloaded_at,
            "note": "failed, nothing saved",
        })
        print(f"{path.name}: download failed ({e})")
        return False
    finally:
        time.sleep(1)  # be polite: at most about one request a second

    path.write_bytes(r.content)
    new_downloads.append({
        "source": source,
        "file": path.name,
        "url": r.url,
        "status": r.status_code,
        "bytes": len(r.content),
        "downloaded_at_utc": downloaded_at,
        "note": "",
    })
    print(f"{path.name}: downloaded {len(r.content):,} bytes")
    return True

## Source 1
For the OECD share price indices we pass `startPeriod`, `endPeriod` and `format="csvfile"` to get the years we need as one CSV file.

In [3]:
OECD_URL = "https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES@DF_FINMARK/.A.SHARE......"
oecd_file = RAW / "oecd_share_prices.csv"

download("OECD", OECD_URL, oecd_file, params={"startPeriod": 2014, "endPeriod": 2024, "format": "csvfile"})

oecd_share_prices.csv: already in data/raw, not downloaded again


True

## Source 2
The World Bank API returns its answer in pages. The first element of each page says how many pages there are, so we keep asking for the
next page until we have them all. Each page is saved and logged as its own file.

In [4]:
WB_URL = "https://api.worldbank.org/v2/country/all/indicator/FP.CPI.TOTL.ZG"
WB_PARAMS = {"format": "json", "date": "2014:2024", "per_page": 1000}

page = 1
pages = 1
while page <= pages:
    path = RAW / f"worldbank_inflation_page_{page:02d}.json"
    if not download("World Bank", WB_URL, path, params={**WB_PARAMS, "page": page}):
        break
    with open(path) as f:
        pages = json.load(f)[0]["pages"]
    page += 1

print(pages, "pages")

worldbank_inflation_page_01.json: already in data/raw, not downloaded again
worldbank_inflation_page_02.json: already in data/raw, not downloaded again
worldbank_inflation_page_03.json: already in data/raw, not downloaded again
3 pages


## Source 3
The Wikipedia page with the ISO 3166-1 country codes is a single HTML page.

In [5]:
WIKI_URL = "https://en.wikipedia.org/wiki/ISO_3166-1"
wiki_file = RAW / "wikipedia_iso_3166_1.html"

download("Wikipedia", WIKI_URL, wiki_file)

wikipedia_iso_3166_1.html: already in data/raw, not downloaded again


True

## Download log
We add this run's downloads to the log that is already there. Then we check that every raw file has a successful download in the log,
so a file can't appear in `data/raw/` without a record of where and when it came from.

In [6]:
if LOG_FILE.exists():
    old_log = pd.read_csv(LOG_FILE)
else:
    old_log = pd.DataFrame(columns=LOG_COLUMNS)

download_log = pd.concat([old_log, pd.DataFrame(new_downloads, columns=LOG_COLUMNS)], ignore_index=True)
download_log.to_csv(LOG_FILE, index=False)
print(len(new_downloads), "new download(s) in this run")
download_log

0 new download(s) in this run


,source,file,url,status,bytes,downloaded_at_utc,note
0,OECD,oecd_share_prices.csv,https://sdmx.oecd.org/public/rest/data/OECD.SD...,NaN,49524,2026-11-16 16:25:01,downloaded before the log recorded times and s...
1,World Bank,worldbank_inflation_page_01.json,https://api.worldbank.org/v2/country/all/indic...,NaN,237345,2026-11-16 16:25:01,downloaded before the log recorded times and s...
2,World Bank,worldbank_inflation_page_02.json,https://api.worldbank.org/v2/country/all/indic...,NaN,227300,2026-11-16 16:25:01,downloaded before the log recorded times and s...
3,World Bank,worldbank_inflation_page_03.json,https://api.worldbank.org/v2/country/all/indic...,NaN,212122,2026-11-16 16:25:01,downloaded before the log recorded times and s...
4,Wikipedia,wikipedia_iso_3166_1.html,https://en.wikipedia.org/wiki/ISO_3166-1,NaN,1031413,2026-11-16 16:25:01,downloaded before the log recorded times and s...


In [7]:
expected = [oecd_file, wiki_file, *sorted(RAW.glob("worldbank_inflation_page_*.json"))]
missing = [p.name for p in expected if not p.exists()]
assert not missing, f"these sources failed to download, see the log above: {missing}"

raw_files = {p.name for p in RAW.iterdir() if p.suffix in {".csv", ".json", ".html"} and p != LOG_FILE}
logged = set(download_log.loc[download_log["bytes"] > 0, "file"])
assert raw_files <= logged, f"raw files without a download in the log: {sorted(raw_files - logged)}"

sizes = download_log[download_log["bytes"] > 0].groupby("file")["bytes"].last()
on_disk = {name: (RAW / name).stat().st_size for name in raw_files}
assert all(sizes[name] == size for name, size in on_disk.items()), "a raw file's size differs from its logged download"
sorted(raw_files)

['oecd_share_prices.csv',
 'wikipedia_iso_3166_1.html',
 'worldbank_inflation_page_01.json',
 'worldbank_inflation_page_02.json',
 'worldbank_inflation_page_03.json']